# Chapter 6 — Regularization for Deep Learning

Runnable companion to `docs/05_regularization.md`. Three experiments:

1. **Overfit demo.** Train an over-parameterized MLP on a *tiny* (1k) MNIST
   subset and watch train accuracy hit 100% while val plateaus far lower.
2. **Add regularization** (dropout, then AdamW weight decay) and watch the
   generalization gap shrink.
3. **Augmentation gallery.** Show what `torchvision.transforms` does to a
   single image, and note which transforms are safe for which task.

Generated by `src/build_chapter_05_regularization.py`.

## Setup

In [1]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = torch.device("cpu")

# Locate the repo root so MNIST is shared across notebooks regardless of cwd.
from pathlib import Path as _Path
_ROOT = _Path.cwd()
while not (_ROOT / "ROADMAP.md").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
MNIST_ROOT = str(_ROOT / "datasets" / "mnist")
print("torch", torch.__version__, "device", DEVICE)
print("mnist root:", MNIST_ROOT)

torch 2.12.0+cu130 device cpu
mnist root: /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/datasets/mnist


## Small MNIST subset

A 1,000-image train set and 1,000-image val set. Small enough that an
over-parameterized model overfits in seconds.

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

mnist_train_full = datasets.MNIST(MNIST_ROOT, train=True,  download=True, transform=transform)
mnist_val_full   = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=transform)

train_subset = Subset(mnist_train_full, list(range(1000)))
val_subset   = Subset(mnist_val_full,   list(range(1000)))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_subset,   batch_size=256, shuffle=False, num_workers=0)
print(f"train: {len(train_subset)} images, val: {len(val_subset)} images")

train: 1000 images, val: 1000 images


## A deliberately over-parameterized MLP

About 1.2M parameters on a 1k-example training set — far more capacity than
data. We expect aggressive overfitting.

In [3]:
class BigMLP(nn.Module):
    def __init__(self, hidden=1024, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),   nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 10),
        )
    def forward(self, x):
        return self.net(x)


def run_one_epoch(model, loader, criterion, optimizer=None):
    if optimizer is not None:
        model.train()
    else:
        model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if optimizer is not None:
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward(); optimizer.step()
        else:
            with torch.no_grad():
                logits = model(x)
                loss = criterion(logits, y)
        loss_sum += loss.item() * y.size(0)
        correct  += (logits.argmax(dim=1) == y).sum().item()
        total    += y.size(0)
    return loss_sum / total, correct / total


def train_and_track(model, num_epochs=15, lr=1e-3, weight_decay=0.0):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(num_epochs):
        tl, ta = run_one_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_one_epoch(model, val_loader,   criterion, None)
        hist["train_loss"].append(tl); hist["val_loss"].append(vl)
        hist["train_acc"].append(ta);  hist["val_acc"].append(va)
    return hist


torch.manual_seed(0)
m_plain = BigMLP(hidden=1024, dropout=0.0).to(DEVICE)
hist_plain = train_and_track(m_plain, num_epochs=15, lr=1e-3, weight_decay=0.0)
print(f"plain MLP    : train_acc={hist_plain['train_acc'][-1]:.3f}, "
      f"val_acc={hist_plain['val_acc'][-1]:.3f}")

/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


plain MLP    : train_acc=1.000, val_acc=0.877


The vanilla model should reach near-100% train accuracy but plateau much
lower on validation — classic overfit shape.

## Add dropout

In [4]:
torch.manual_seed(0)
m_dropout = BigMLP(hidden=1024, dropout=0.5).to(DEVICE)
hist_dropout = train_and_track(m_dropout, num_epochs=15, lr=1e-3, weight_decay=0.0)
print(f"+ dropout p=0.5: train_acc={hist_dropout['train_acc'][-1]:.3f}, "
      f"val_acc={hist_dropout['val_acc'][-1]:.3f}")

+ dropout p=0.5: train_acc=0.986, val_acc=0.861


## Add AdamW weight decay (no dropout)

In [5]:
torch.manual_seed(0)
m_wd = BigMLP(hidden=1024, dropout=0.0).to(DEVICE)
hist_wd = train_and_track(m_wd, num_epochs=15, lr=1e-3, weight_decay=1e-2)
print(f"+ weight_decay=1e-2: train_acc={hist_wd['train_acc'][-1]:.3f}, "
      f"val_acc={hist_wd['val_acc'][-1]:.3f}")

+ weight_decay=1e-2: train_acc=1.000, val_acc=0.877


## Dropout + weight decay together

In [6]:
torch.manual_seed(0)
m_both = BigMLP(hidden=1024, dropout=0.5).to(DEVICE)
hist_both = train_and_track(m_both, num_epochs=15, lr=1e-3, weight_decay=1e-2)
print(f"+ dropout + weight_decay: train_acc={hist_both['train_acc'][-1]:.3f}, "
      f"val_acc={hist_both['val_acc'][-1]:.3f}")

+ dropout + weight_decay: train_acc=0.989, val_acc=0.856


## Compare the four runs

In [7]:
runs = {
    "plain":             hist_plain,
    "+ dropout":         hist_dropout,
    "+ weight decay":    hist_wd,
    "+ both":            hist_both,
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, h in runs.items():
    axes[0].plot(h["val_loss"],  label=name)
    axes[1].plot(h["val_acc"],   label=name)
axes[0].set_title("Validation loss");    axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print()
print("Final validation accuracy:")
for name, h in runs.items():
    gap = h["train_acc"][-1] - h["val_acc"][-1]
    print(f"  {name:18s} val={h['val_acc'][-1]:.3f}  train-val gap={gap:.3f}")


Final validation accuracy:
  plain              val=0.877  train-val gap=0.123
  + dropout          val=0.861  train-val gap=0.125
  + weight decay     val=0.877  train-val gap=0.123
  + both             val=0.856  train-val gap=0.133


/tmp/ipykernel_185780/340789765.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The train-val gap is the most direct measure of overfit. Both dropout and
weight decay shrink it; the combination usually wins on small datasets.

## Augmentation gallery — what each transform does

Visualize the effect of common `torchvision.transforms` on a single MNIST
digit. Pick the ones that *do not change the label*. For example,
horizontal flip is *not* safe on digits.

In [8]:
raw_ds = datasets.MNIST(MNIST_ROOT, train=True, download=False, transform=None)
img, label = raw_ds[7]
print(f"original label: {label}")

augs = {
    "original":                  (lambda p: p),
    "RandomRotation(15)":        transforms.RandomRotation(15),
    "RandomAffine(translate)":   transforms.RandomAffine(0, translate=(0.1, 0.1)),
    "RandomResizedCrop(28)":     transforms.RandomResizedCrop(28, scale=(0.8, 1.0)),
    "ColorJitter(0.5)":          transforms.ColorJitter(brightness=0.5, contrast=0.5),
    "HorizontalFlip (UNSAFE!)":  transforms.RandomHorizontalFlip(p=1.0),
}

fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
torch.manual_seed(0)
for ax, (name, tfm) in zip(axes, augs.items()):
    ax.imshow(tfm(img), cmap="gray")
    ax.set_title(name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()
print("RandomHorizontalFlip is unsafe for digits — a flipped '7' is not a '7'.")

original label: 3
RandomHorizontalFlip is unsafe for digits — a flipped '7' is not a '7'.


/tmp/ipykernel_185780/932315257.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Label smoothing — calibration comparison

`CrossEntropyLoss(label_smoothing=0.1)` replaces the one-hot target with a
softer distribution. The model becomes *less confident* and usually
better-calibrated.

Compare the *output probability* of the predicted class on val, with and
without label smoothing.

In [9]:
def train_with_loss(loss_fn, num_epochs=10):
    torch.manual_seed(0)
    m = BigMLP(hidden=512, dropout=0.0).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-2)
    for _ in range(num_epochs):
        m.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(m(x), y)
            loss.backward(); opt.step()
    m.eval()
    confidences = []
    with torch.no_grad():
        for x, _ in val_loader:
            x = x.to(DEVICE)
            probs = F.softmax(m(x), dim=-1)
            confidences.extend(probs.max(dim=-1).values.tolist())
    return np.array(confidences)


conf_no_ls = train_with_loss(nn.CrossEntropyLoss())
conf_ls    = train_with_loss(nn.CrossEntropyLoss(label_smoothing=0.1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(conf_no_ls, bins=30, alpha=0.5, label=f"no LS  (mean={conf_no_ls.mean():.3f})")
ax.hist(conf_ls,    bins=30, alpha=0.5, label=f"LS=0.1 (mean={conf_ls.mean():.3f})")
ax.set_xlabel("max softmax probability on val"); ax.set_ylabel("count")
ax.set_title("Effect of label smoothing on predicted-class confidence")
ax.legend(); plt.tight_layout(); plt.show()

/tmp/ipykernel_185780/1383196383.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); plt.tight_layout(); plt.show()


With label smoothing, the predicted-class probability concentrates lower
(closer to `1 - smoothing`). The model is now less likely to make
over-confident wrong predictions — useful when calibration matters
(active learning, selective prediction, ensembling).

## Self-test

<details>
<summary>Q1 — Why is RandomHorizontalFlip unsafe on MNIST?</summary>

Many digits are not horizontally symmetric — a flipped `7` is not still a
`7`, and a flipped `2` looks like nothing in particular. Augmentations
must preserve the label.
</details>

<details>
<summary>Q2 — Why prefer AdamW over Adam(weight_decay=...)?</summary>

`Adam(weight_decay)` mixes the L2 penalty into the gradient *before* the
adaptive per-parameter learning rate, so frequently-updated parameters get
under-decayed. `AdamW` applies a separate uniform decay step.
</details>